# 02 - Preprocessing

Este notebook documenta el preprocesamiento del dataset antes del modelado.

Objetivos:
- Separar variables numericas y categoricas.
- Definir pipeline de transformacion reproducible.
- Preparar datos para entrenamiento y evaluacion.

In [ ]:
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
# Carga de datos

DATA_PATH = Path("../data/train.csv")
df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
# Definicion de X e y

target = "SalePrice"
X = df.drop(columns=[target]).copy()
y = df[target].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
# Identificacion de columnas numericas y categoricas

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print(f"Num features: {len(numeric_features)}")
print(f"Cat features: {len(categorical_features)}")

print("Ejemplo numericas:", numeric_features[:10])
print("Ejemplo categoricas:", categorical_features[:10])

In [ ]:
# Pipelines de transformacion

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

preprocessor

In [ ]:
# Split train/test y transformacion de ejemplo

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("X_train original:", X_train.shape)
print("X_train procesado:", X_train_processed.shape)
print("X_test original:", X_test.shape)
print("X_test procesado:", X_test_processed.shape)

## Notas para el informe

- Los nulos numericos se imputan con mediana.
- Los nulos categoricos se imputan con moda.
- Las categoricas se codifican con One-Hot Encoding.
- Las numericas se estandarizan con StandardScaler.
- Se usa `train_test_split` para evitar leakage.